# Atelier 1.E — Comparaison de modèles

**Durée : 40 min**

Quel modèle choisir pour trier les demandes de Fictiva ? Vous comparez trois modèles open-weight de familles différentes sur la tâche de l'atelier 1.C, selon quatre critères : **qualité, coût, latence, confidentialité**.

In [ ]:
import outils_llm as llm

MODELE = llm.choisir_modele()  # ou forcez, par ex. "nvidia:nvidia/nemotron-3.5-lightning-30b-a3b", "groq:openai/gpt-oss-120b"
N_MAX = 20                     # les paliers gratuits limitent le débit ; None pour tout le jeu de test
print("Modèle utilisé :", MODELE)

import pandas as pd

## 1. Les candidats

Des modèles de tailles et de familles différentes, chez des fournisseurs différents. Adaptez la liste à vos accès : seuls les modèles dont le fournisseur est configuré sont retenus, dans l'ordre de la liste, jusqu'à trois.

Avec OpenRouter sans crédit acheté, les modèles « :free » sont limités à 50 requêtes par jour : réduisez `N_MAX` si besoin.

In [ ]:
CANDIDATS = [
    "groq:openai/gpt-oss-20b",                       # raisonnement, 20 milliards de paramètres
    "nvidia:nvidia/nemotron-3.5-lightning-30b-a3b",  # Nemotron 3.5, 30 milliards dont 3 actifs (MoE)
    "groq:qwen/qwen3.8-27b",                         # Qwen 3.8, 27 milliards
    "groq:openai/gpt-oss-120b",                      # raisonnement, 120 milliards (MoE)
    "openrouter:google/gemma-4-26b-a4b-it:free",     # Gemma 4, 26 milliards (MoE), variante gratuite
]
disponibles = llm.fournisseurs_disponibles()
MODELES = [m for m in CANDIDATS if disponibles[llm.decouper(m)[0]]][:3]
MODELES

## 2. La tâche

Même prompt et même schéma pour tous les modèles : le prompt enregistré à l'atelier 1.C, ou à défaut le prompt de référence fourni.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Categorie = Literal["reclamation", "remboursement", "litige_facturation", "question_produit", "autre"]
Urgence = Literal["haute", "normale", "basse"]

class Tri(BaseModel):
    justification: str = Field(description="Une phrase citant l'élément décisif de la demande")
    categorie: Categorie
    urgence: Urgence

try:
    PROMPT = llm.charger_prompt("tri_demandes")
    print("Prompt de l'atelier 1.C")
except FileNotFoundError:
    PROMPT = llm.charger_prompt("tri_reference")
    print("Prompt de référence")

df_test = llm.charger_demandes("test")
echantillon = df_test if N_MAX is None else df_test.head(N_MAX)
lignes = [ligne for _, ligne in echantillon.iterrows()]

## 3. Mesurer qualité, latence et coût

In [ ]:
def evaluer_modele(modele):
    reponses = llm.en_lot(
        lambda ligne: llm.demander(llm.formater_demande(ligne), systeme=PROMPT, modele=modele, schema=Tri),
        lignes,
    )
    predictions = [r.objet if isinstance(r, llm.Reponse) else None for r in reponses]
    bonne_categorie = [p is not None and p.categorie == v for p, v in zip(predictions, echantillon.categorie)]
    bonne_urgence = [p is not None and p.urgence == v for p, v in zip(predictions, echantillon.urgence)]
    structures = {r.structure for r in reponses if isinstance(r, llm.Reponse)}
    return {
        "modele": modele,
        "exactitude_categorie": round(sum(bonne_categorie) / len(lignes), 3),
        "exactitude_urgence": round(sum(bonne_urgence) / len(lignes), 3),
        "exactitude_globale": round(sum(c and u for c, u in zip(bonne_categorie, bonne_urgence)) / len(lignes), 3),
        "echecs": sum(p is None for p in predictions),
        "structure": ", ".join(sorted(s for s in structures if s)),
        **llm.bilan(reponses),
    }, predictions

resultats, predictions = {}, {}
for modele in MODELES:
    print("→", modele)
    resultats[modele], predictions[modele] = evaluer_modele(modele)

comparaison = pd.DataFrame(resultats.values()).set_index("modele")
comparaison

`cout_pour_1000_usd` est estimé au tarif public payant (`llm.TARIFS`) : les paliers gratuits ne sont pas facturés, mais ils ne conviennent pas à la production.

## 4. Stabilité

Un même modèle donne-t-il la même réponse deux fois ? Relançons le premier modèle sur les dix premières demandes.

In [ ]:
modele = MODELES[0]
premiers = lignes[:10]
second_passage = llm.en_lot(
    lambda ligne: llm.demander(llm.formater_demande(ligne), systeme=PROMPT, modele=modele, schema=Tri),
    premiers,
)
identiques = sum(
    a is not None and isinstance(b, llm.Reponse) and b.objet is not None
    and (a.categorie, a.urgence) == (b.objet.categorie, b.objet.urgence)
    for a, b in zip(predictions[modele][:10], second_passage)
)
print(f"{modele} : {identiques}/{len(premiers)} classements identiques entre deux passages")

## 5. Décider

La qualité ne suffit pas : il faut pondérer les critères selon le contexte. La confidentialité dépend de l'hébergement, des conditions d'utilisation (les paliers gratuits peuvent conserver ou exploiter les données envoyées) et de la localisation des données. Les notes ci-dessous sont des **appréciations à discuter**, pas des mesures.

Deux scénarios :
- **A.** 300 demandes par jour, données clients ordinaires, budget serré ;
- **B.** demandes contenant des données de santé (clinique cliente), exigences nLPD strictes.

In [ ]:
CONFIDENTIALITE = {"nvidia": 0.5, "groq": 0.5, "huggingface": 0.5, "openrouter": 0.4}  # à discuter

def score(comparaison, poids):
    t = comparaison.copy()
    cout = t["cout_pour_1000_usd"].astype(float)
    t["note_qualite"] = t["exactitude_globale"]
    t["note_cout"] = 1 - cout / cout.max() if cout.max() > 0 else 1.0
    t["note_latence"] = t["latence_moyenne_s"].min() / t["latence_moyenne_s"]
    t["note_confidentialite"] = [CONFIDENTIALITE[llm.decouper(m)[0]] for m in t.index]
    t["score"] = sum(poids[critere] * t[f"note_{critere}"].fillna(0) for critere in poids)
    return t[["note_qualite", "note_cout", "note_latence", "note_confidentialite", "score"]].round(3).sort_values("score", ascending=False)

✏️ Les poids ci-dessous sont un exemple : modifiez-les, comparez les classements et rédigez votre propre recommandation.

In [ ]:
POIDS_A = {"qualite": 0.5, "cout": 0.3, "latence": 0.1, "confidentialite": 0.1}
POIDS_B = {"qualite": 0.4, "cout": 0.05, "latence": 0.05, "confidentialite": 0.5}

display(score(comparaison, POIDS_A))
display(score(comparaison, POIDS_B))

RECOMMANDATION = """Scénario A : le modèle le plus économique dont l'exactitude reste proche du meilleur.
À 300 demandes par jour, le coût mensuel reste faible pour tous les modèles open-weight hébergés :
un écart de quelques points d'exactitude pèse plus lourd que quelques dollars. Le palier gratuit
sert à prototyper ; la production passe par une offre payante, avec ses garanties contractuelles.
Scénario B : un modèle open-weight auto-hébergé sur une infrastructure maîtrisée, ou un fournisseur
qui garantit l'hébergement en Suisse ou dans l'UE et l'absence de rétention, validé par le délégué à la
protection des données. À défaut, anonymiser les demandes avant envoi."""

## À retenir

- On compare des modèles **sur sa propre tâche**, avec son propre jeu de test : les classements publics ne suffisent pas.
- Le choix pondère qualité, coût, latence et confidentialité ; les poids dépendent du contexte métier.
- Les modèles open-weight peuvent tourner chez plusieurs fournisseurs, ou sur sa propre infrastructure : on peut changer d'hébergement sans changer de modèle.
- Le plus gros modèle n'est pas toujours le meilleur choix : pour une tâche simple et bien cadrée, un modèle plus léger peut suffire.